# Models for Future value prediction and to find if its a good investment

### Library Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import (LinearRegression,Ridge,Lasso,LogisticRegression)
from sklearn.ensemble import (RandomForestRegressor,GradientBoostingRegressor,RandomForestClassifier,GradientBoostingClassifier)
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor,XGBClassifier
from lightgbm import LGBMRegressor,LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix,
    precision_score,
    f1_score,
    recall_score,
    roc_curve,
    roc_auc_score,
    accuracy_score
)
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import mlflow.xgboost
import mlflow.lightgbm

In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

### Helper Functions

In [ ]:
def scale_features(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train),
                                   columns=X_train.columns, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test),
                                  columns=X_test.columns, index=X_test.index)
    return X_train_scaled, X_test_scaled, scaler


def log_model_by_type(model, name, signature):
    if name == "XGBoost":
        mlflow.xgboost.log_model(model, name="model", signature=signature)
    elif name == "LightGBM":
        mlflow.lightgbm.log_model(model, name="model", signature=signature)
    else:
        mlflow.sklearn.log_model(model, name="model", signature=signature)


def regression_metrics(y_true, preds):
    return {
        "rmse": np.sqrt(mean_squared_error(y_true, preds)),
        "mae": mean_absolute_error(y_true, preds),
        "r2": r2_score(y_true, preds),
    }


def classification_metrics(y_true, preds, proba):
    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds),
        "recall": recall_score(y_true, preds),
        "f1": f1_score(y_true, preds),
        "roc_auc": roc_auc_score(y_true, proba),
    }

### Data Load and Train Test Split

In [3]:
df = pd.read_csv(r'Datasets/india_housing_prices_with_target_columns_encoded.csv')
df.head()

,State,Locality,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,...,Facing_East,Facing_North,Facing_South,Facing_West,Furnished_Status_Furnished,Furnished_Status_Semi_Furnished,Furnished_Status_Unfurnished,Owner_Type_Broker,Owner_Type_Builder,Owner_Type_Owner
0,Tamil Nadu,Locality_84,1,4740,489.76,0.10,1990,22,1,35,...,0,0,0,1,1,0,0,0,0,1
1,Maharashtra,Locality_490,3,2364,195.52,0.08,2008,21,20,17,...,0,1,0,0,0,0,1,0,1,0
2,Punjab,Locality_167,2,3642,183.79,0.05,1997,19,27,28,...,0,0,1,0,0,1,0,1,0,0
3,Rajasthan,Locality_393,2,2741,300.29,0.11,1991,21,26,34,...,0,1,0,0,1,0,0,0,1,0
4,Rajasthan,Locality_466,4,4823,182.90,0.04,2002,3,2,23,...,1,0,0,0,0,1,0,0,1,0


### Regression Model

In [4]:
x_reg_cols = ['BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Floor_No', 'Total_Floors',
    'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals',
    'Public_Transport_Accessibility', 'Parking_Space', 'Security',
    'Availability_Status', 'Amenities_Count',
    'City_Ahmedabad', 'City_Amritsar', 'City_Bangalore', 'City_Bhopal',
    'City_Bhubaneswar', 'City_Bilaspur', 'City_Chennai', 'City_Coimbatore',
    'City_Cuttack', 'City_Dehradun', 'City_Durgapur', 'City_Dwarka',
    'City_Faridabad', 'City_Gaya', 'City_Gurgaon', 'City_Guwahati',
    'City_Haridwar', 'City_Hyderabad', 'City_Indore', 'City_Jaipur',
    'City_Jamshedpur', 'City_Jodhpur', 'City_Kochi', 'City_Kolkata',
    'City_Lucknow', 'City_Ludhiana', 'City_Mangalore', 'City_Mumbai',
    'City_Mysore', 'City_Nagpur', 'City_New_Delhi', 'City_Noida',
    'City_Patna', 'City_Pune', 'City_Raipur', 'City_Ranchi', 'City_Silchar',
    'City_Surat', 'City_Trivandrum', 'City_Vijayawada',
    'City_Vishakhapatnam', 'City_Warangal', 'Property_Type_Apartment',
    'Property_Type_Independent_House', 'Property_Type_Villa', 'Facing_East',
    'Facing_North', 'Facing_South', 'Facing_West',
    'Furnished_Status_Furnished', 'Furnished_Status_Semi_Furnished',
    'Furnished_Status_Unfurnished', 'Owner_Type_Broker',
    'Owner_Type_Builder', 'Owner_Type_Owner']
X_reg = df[x_reg_cols]
y_reg = df['Future_Price_5Y']

In [5]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

In [ ]:
X_train_reg_scaled, X_test_reg_scaled, scaler = scale_features(X_train_reg, X_test_reg)

In [8]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "RandomForest": RandomForestRegressor(random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
    "LightGBM": LGBMRegressor(random_state=42),
}

In [9]:

mlflow.set_experiment("real_estate_regression")

<Experiment: artifact_location='mlflow-artifacts:/4', creation_time=1787666057911, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1787666057911, lifecycle_stage='active', name='real_estate_regression', tags={}, trace_location=None, workspace='default'>

In [ ]:
def train_and_log_regressor(models):
    for name, model in models.items():
        with mlflow.start_run(run_name=name):
            model.fit(X_train_reg_scaled, y_train_reg)
            preds = model.predict(X_test_reg_scaled)

            metrics = regression_metrics(y_test_reg, preds)

            mlflow.log_param("model_type", name)
            mlflow.log_metrics(metrics)

            signature = infer_signature(X_test_reg_scaled, preds)
            log_model_by_type(model, name, signature)

            print(f"{name}: RMSE={metrics['rmse']:.2f}  MAE={metrics['mae']:.2f}  R2={metrics['r2']:.3f}")

In [25]:
train_and_log_regressor(models=models)

LinearRegression: RMSE=39.05  MAE=29.32  R2=0.960
🏃 View run LinearRegression at: http://127.0.0.1:5000/#/experiments/5/runs/3af2c8f4941f48dd91af5bb52dab6e04
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
Ridge: RMSE=39.05  MAE=29.32  R2=0.960
🏃 View run Ridge at: http://127.0.0.1:5000/#/experiments/5/runs/2777ce5f26cc400a8f12c817d28465e1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
Lasso: RMSE=39.57  MAE=29.08  R2=0.959
🏃 View run Lasso at: http://127.0.0.1:5000/#/experiments/5/runs/9f72b5a8d79d466194a9401e59ab0b0d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
RandomForest: RMSE=35.43  MAE=24.71  R2=0.967
🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/5/runs/023bd4a9648f4ce2a7ad2e1812699818
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
GradientBoosting: RMSE=37.87  MAE=26.61  R2=0.963
🏃 View run GradientBoosting at: http://127.0.0.1:5000/#/experiments/5/runs/0d5c6f0f80e74f2199dc8ffe720f5008
🧪 View experiment at: 

In [11]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [-1, 5, 10, 20],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "num_leaves": [31, 50, 70, 100],
    "subsample": [0.7, 0.8, 1.0],
}

lgbm = LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1)

search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring="r2",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train_reg_scaled, y_train_reg)

print("Best params:", search.best_params_)
print("Best CV R2:", round(search.best_score_, 4))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'subsample': 0.7, 'num_leaves': 31, 'n_estimators': 200, 'max_depth': -1, 'learning_rate': 0.05}
Best CV R2: 0.9714


In [ ]:
best_lgbm = search.best_estimator_

preds = best_lgbm.predict(X_test_reg_scaled)
metrics = regression_metrics(y_test_reg, preds)

print(f"Tuned LightGBM (test): RMSE={metrics['rmse']:.2f}  MAE={metrics['mae']:.2f}  R2={metrics['r2']:.3f}")

with mlflow.start_run(run_name="LightGBM_tuned"):
    mlflow.log_params(search.best_params_)
    mlflow.log_metrics(metrics)
    signature = infer_signature(X_test_reg_scaled, preds)
    log_model_by_type(best_lgbm, "LightGBM", signature)

In [13]:
mlflow.register_model(
    "runs:/9d7b35bb10a14de09c466de523c2df6a/model",
    "real_estate_regression_model"
)

Registered model 'real_estate_regression_model' already exists. Creating a new version of this model...
2026/08/26 11:23:29 WARNING mlflow.tracking._model_registry.fluent: Run with id 9d7b35bb10a14de09c466de523c2df6a has no artifacts at artifact path 'model', registering model based on models:/m-0961365c37104ca99cc637cbd969368b instead
2026/08/26 11:23:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: real_estate_regression_model, version 2
Created version '2' of model 'real_estate_regression_model'.


<ModelVersion: aliases=[], creation_timestamp=1787723609148, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1787723609148, metrics=None, model_id=None, name='real_estate_regression_model', params=None, run_id='9d7b35bb10a14de09c466de523c2df6a', run_link='', source='models:/m-0961365c37104ca99cc637cbd969368b', status='READY', status_message=None, tags={}, user_id='', version='2', workspace='default'>

### Classification Model

In [36]:
x_clf_cols = ['Size_in_SqFt', 'Price_in_Lakhs', 'Floor_No', 'Total_Floors',
    'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals',
    'Public_Transport_Accessibility', 'Security',
    'City_Ahmedabad', 'City_Amritsar', 'City_Bangalore', 'City_Bhopal',
    'City_Bhubaneswar', 'City_Bilaspur', 'City_Chennai', 'City_Coimbatore',
    'City_Cuttack', 'City_Dehradun', 'City_Durgapur', 'City_Dwarka',
    'City_Faridabad', 'City_Gaya', 'City_Gurgaon', 'City_Guwahati',
    'City_Haridwar', 'City_Hyderabad', 'City_Indore', 'City_Jaipur',
    'City_Jamshedpur', 'City_Jodhpur', 'City_Kochi', 'City_Kolkata',
    'City_Lucknow', 'City_Ludhiana', 'City_Mangalore', 'City_Mumbai',
    'City_Mysore', 'City_Nagpur', 'City_New_Delhi', 'City_Noida',
    'City_Patna', 'City_Pune', 'City_Raipur', 'City_Ranchi', 'City_Silchar',
    'City_Surat', 'City_Trivandrum', 'City_Vijayawada',
    'City_Vishakhapatnam', 'City_Warangal', 'Property_Type_Apartment',
    'Property_Type_Independent_House', 'Property_Type_Villa', 'Facing_East',
    'Facing_North', 'Facing_South', 'Facing_West',
    'Furnished_Status_Furnished', 'Furnished_Status_Semi_Furnished',
    'Furnished_Status_Unfurnished', 'Owner_Type_Broker',
    'Owner_Type_Builder', 'Owner_Type_Owner']
X_clf = df[x_clf_cols]
y_clf = df['Good_Investment']

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

In [ ]:
X_train_clf_scaled, X_test_clf_scaled, scaler_clf = scale_features(X_train_clf, X_test_clf)

In [16]:
mlflow.set_experiment("real_estate_classification")

clf_models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1),
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
}

In [ ]:
def train_and_log_classifier(clf_models):
    for name, model in clf_models.items():
        with mlflow.start_run(run_name=name):
            model.fit(X_train_clf_scaled, y_train_clf)
            preds = model.predict(X_test_clf_scaled)
            proba = model.predict_proba(X_test_clf_scaled)[:, 1]

            metrics = classification_metrics(y_test_clf, preds, proba)

            mlflow.log_param("model_type", name)
            mlflow.log_metrics(metrics)

            signature = infer_signature(X_test_clf_scaled, preds)
            log_model_by_type(model, name, signature)

            print(f"{name}: Acc={metrics['accuracy']:.3f} Prec={metrics['precision']:.3f} "
                  f"Rec={metrics['recall']:.3f} F1={metrics['f1']:.3f} AUC={metrics['roc_auc']:.3f}")

In [26]:
train_and_log_classifier(clf_models=clf_models)

LogisticRegression: Acc=0.681 Prec=0.450 Rec=0.713 F1=0.552 AUC=0.723
🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/5/runs/df296296a738418b94fd9f5cbbbbe105
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
KNN: Acc=0.684 Prec=0.365 Rec=0.197 F1=0.256 AUC=0.576
🏃 View run KNN at: http://127.0.0.1:5000/#/experiments/5/runs/27c638a067b24c2e88ee1df1e26197e9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
DecisionTree: Acc=0.659 Prec=0.386 Rec=0.401 F1=0.393 AUC=0.579
🏃 View run DecisionTree at: http://127.0.0.1:5000/#/experiments/5/runs/26f0fe69a03f4fdf9ae4b148bd1e14c2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
RandomForest: Acc=0.685 Prec=0.445 Rec=0.579 F1=0.503 AUC=0.722
🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/5/runs/69b0576c1de24cd6b3bce98d5af33cc9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
GradientBoosting: Acc=0.723 Prec=0.428 Rec=0.015 F1=0.029 AUC=0.722
🏃 View run GradientBoostin

In [29]:
mlflow.set_experiment("real_estate_classification")
best_clf_models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "KNN": KNeighborsClassifier(), 
    "DecisionTree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1, scale_pos_weight=2.57),
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, class_weight="balanced"),
}

In [30]:
train_and_log_classifier(clf_models=best_clf_models)

LogisticRegression: Acc=0.681 Prec=0.450 Rec=0.713 F1=0.552 AUC=0.723
🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/5/runs/f5d46c6de19a4ff1837c6f99d4c14937
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
KNN: Acc=0.684 Prec=0.365 Rec=0.197 F1=0.256 AUC=0.576
🏃 View run KNN at: http://127.0.0.1:5000/#/experiments/5/runs/dfd54402d840454d8d2c57eb516440bf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
DecisionTree: Acc=0.659 Prec=0.386 Rec=0.401 F1=0.393 AUC=0.579
🏃 View run DecisionTree at: http://127.0.0.1:5000/#/experiments/5/runs/8ea7756a807449c4a5fb2950926810a0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
RandomForest: Acc=0.685 Prec=0.445 Rec=0.579 F1=0.503 AUC=0.722
🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/5/runs/a192eec824ac4a0a8dd25075f07541f9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
GradientBoosting: Acc=0.723 Prec=0.428 Rec=0.015 F1=0.029 AUC=0.722
🏃 View run GradientBoostin

In [19]:
best_clf = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, class_weight="balanced")
best_clf.fit(X_train_clf_scaled, y_train_clf)
proba = best_clf.predict_proba(X_test_clf_scaled)[:, 1]

In [20]:
from sklearn.metrics import precision_recall_curve

thresholds = np.arange(0.1, 0.9, 0.05)
results = []
for t in thresholds:
    preds_t = (proba >= t).astype(int)
    f1_t = f1_score(y_test_clf, preds_t)
    results.append((t, f1_t))
    print(f"threshold={t:.2f}  F1={f1_t:.3f}")

best_threshold = max(results, key=lambda x: x[1])[0]
print("\nBest threshold:", best_threshold)

threshold=0.10  F1=0.432
threshold=0.15  F1=0.433
threshold=0.20  F1=0.459
threshold=0.25  F1=0.557
threshold=0.30  F1=0.569
threshold=0.35  F1=0.575
threshold=0.40  F1=0.578
threshold=0.45  F1=0.580
threshold=0.50  F1=0.580
threshold=0.55  F1=0.580
threshold=0.60  F1=0.577
threshold=0.65  F1=0.564
threshold=0.70  F1=0.057
threshold=0.75  F1=0.003
threshold=0.80  F1=0.000
threshold=0.85  F1=0.000

Best threshold: 0.5000000000000001


In [51]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "num_leaves": [31, 50, 70, 100],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "max_depth": [-1, 5, 10, 20],
    "subsample": [0.7, 0.8, 1.0],
}

lgbm_clf = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1, class_weight="balanced")

search_clf = RandomizedSearchCV(
    estimator=lgbm_clf,
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring="f1",    
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search_clf.fit(X_train_clf_scaled, y_train_clf)
print("Best params:", search_clf.best_params_)
print("Best CV F1:", round(search_clf.best_score_, 4))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'subsample': 0.8, 'num_leaves': 70, 'n_estimators': 100, 'max_depth': -1, 'learning_rate': 0.05}
Best CV F1: 0.5815


In [ ]:
best_clf_tuned = search_clf.best_estimator_

preds = best_clf_tuned.predict(X_test_clf_scaled)
proba = best_clf_tuned.predict_proba(X_test_clf_scaled)[:, 1]

metrics = classification_metrics(y_test_clf, preds, proba)

print(f"Tuned LightGBM (test): Acc={metrics['accuracy']:.3f} Prec={metrics['precision']:.3f} "
      f"Rec={metrics['recall']:.3f} F1={metrics['f1']:.3f} AUC={metrics['roc_auc']:.3f}")

with mlflow.start_run(run_name="LightGBM_clf_tuned"):
    mlflow.log_params(search_clf.best_params_)
    mlflow.log_metrics(metrics)
    signature = infer_signature(X_test_clf_scaled, preds)
    log_model_by_type(best_clf_tuned, "LightGBM", signature)

In [52]:
mlflow.register_model(
    "runs:/e56c05a0b2004021895bc89e911ce255/model",
    "real_estate_classification_model"
)

Registered model 'real_estate_classification_model' already exists. Creating a new version of this model...
2026/08/26 12:17:34 WARNING mlflow.tracking._model_registry.fluent: Run with id e56c05a0b2004021895bc89e911ce255 has no artifacts at artifact path 'model', registering model based on models:/m-03ece1ba70c04e4a84591a946b47133f instead
2026/08/26 12:17:34 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: real_estate_classification_model, version 3
Created version '3' of model 'real_estate_classification_model'.


<ModelVersion: aliases=[], creation_timestamp=1787726854177, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1787726854177, metrics=None, model_id=None, name='real_estate_classification_model', params=None, run_id='e56c05a0b2004021895bc89e911ce255', run_link='', source='models:/m-03ece1ba70c04e4a84591a946b47133f', status='READY', status_message=None, tags={}, user_id='', version='3', workspace='default'>

### Classification Experiments

In [ ]:
#df['size_per_floor'] = df['Size_in_SqFt'] / df['Total_Floors']
#df['school_hospital_sum'] = df['Nearby_Schools'] + df['Nearby_Hospitals']

In [ ]:
#x_clf_cols = x_clf_cols + ['size_per_floor', 'school_hospital_sum']
#X_clf = df[x_clf_cols]

In [ ]:
#mlflow.set_experiment("real_estate_classification")

<Experiment: artifact_location='mlflow-artifacts:/5', creation_time=1787668223249, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1787668223249, lifecycle_stage='active', name='real_estate_classification', tags={}, trace_location=None, workspace='default'>

In [ ]:
#X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
#    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

In [ ]:
#scaler_clf = StandardScaler()
#X_train_clf_scaled = pd.DataFrame(scaler_clf.fit_transform(X_train_clf),
#                                  columns=X_train_clf.columns, index=X_train_clf.index)
#X_test_clf_scaled = pd.DataFrame(scaler_clf.transform(X_test_clf),
#                                 columns=X_test_clf.columns, index=X_test_clf.index)

In [ ]:
#train_and_log_classifier(best_clf_models)

LogisticRegression: Acc=0.681 Prec=0.450 Rec=0.713 F1=0.552 AUC=0.723
🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/5/runs/dfcb760190c443cda0ea29dfedb9ac3c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
KNN: Acc=0.684 Prec=0.363 Rec=0.196 F1=0.255 AUC=0.570
🏃 View run KNN at: http://127.0.0.1:5000/#/experiments/5/runs/e381a6066e204e92b5999032f46bd0f1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
DecisionTree: Acc=0.657 Prec=0.381 Rec=0.395 F1=0.388 AUC=0.576
🏃 View run DecisionTree at: http://127.0.0.1:5000/#/experiments/5/runs/124b3cbad7504bccb773d92fe3002cb6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
RandomForest: Acc=0.689 Prec=0.451 Rec=0.589 F1=0.511 AUC=0.723
🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/5/runs/09f5f5007376499cbcc7af6427694052
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
GradientBoosting: Acc=0.723 Prec=0.419 Rec=0.016 F1=0.031 AUC=0.721
🏃 View run GradientBoostin

In [ ]:
#x_clf_cols = [c for c in x_clf_cols if c not in ['size_per_floor', 'school_hospital_sum']]
#len(x_clf_cols) 

64